# 02 — Feature Engineering

Building on `01_data_exploration.ipynb`, this notebook engineers additional features for the XGBoost demand forecasting model. The dataset is weekly, per-medicine time series data (150 medicines × 156 weeks, 2023–2025), so features are built **per medicine** and using only **past information** (no leakage from the `next_week_demand` target).

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Dataset/smart_pharmacy_dataset.csv")
df = df.sort_values(["medicine_id", "year", "week_number"]).reset_index(drop=True)

original_cols = df.columns.tolist()
df.shape

(23400, 19)

## 1. Cyclical Time Features

`week_number` (1–52) is a linear feature, but seasonality is cyclical — week 52 is adjacent to week 1, not far from it. Sine/cosine encoding represents this properly so the model can learn smooth seasonal patterns instead of an artificial cliff between week 52 and week 1. `season` and `is_holiday_week` already give the model coarse seasonal signal, so these add finer-grained periodicity on top.

In [2]:
df["week_sin"] = np.sin(2 * np.pi * df["week_number"] / 52)
df["week_cos"] = np.cos(2 * np.pi * df["week_number"] / 52)

df[["week_number", "week_sin", "week_cos"]].head()

,week_number,week_sin,week_cos
0,1,0.120537,0.992709
1,2,0.239316,0.970942
2,3,0.354605,0.935016
3,4,0.464723,0.885456
4,5,0.568065,0.822984


## 2. Categorical Encoding

- `medicine_importance` (Low / Medium / High / Critical) is ordinal, so it's mapped to an integer score 0–3 rather than one-hot encoded — this preserves the ordering, which is meaningful (Critical > High > Medium > Low).
- `category` (8 drug categories) and `season` (4 seasons) are unordered, so they're one-hot encoded. Tree-based models like XGBoost don't need a reference level dropped — all dummy columns are kept.

In [3]:
importance_map = {"Low": 0, "Medium": 1, "High": 2, "Critical": 3}
df["medicine_importance_score"] = df["medicine_importance"].map(importance_map)

df = pd.get_dummies(df, columns=["category", "season"], prefix=["cat", "season"])

[c for c in df.columns if c.startswith("cat_") or c.startswith("season_")]

['cat_Antibiotic',
 'cat_Cardiac',
 'cat_Diabetes',
 'cat_Emergency',
 'cat_Fever/Pain Relief',
 'cat_Gastrointestinal',
 'cat_Respiratory',
 'cat_Vitamin/Supplement',
 'season_Monsoon',
 'season_Normal',
 'season_Summer',
 'season_Winter']

## 3. Stock Health Features

These translate raw stock numbers into the kind of signal a pharmacist actually reasons with — *how many weeks of cover do I have left?* and *did I already run short last week?*

- `weeks_of_stock`: current stock divided by recent average weekly sales — a runway estimate.
- `stockout_flag`: 1 if last week's stock on hand was already below what was sold — a signal that this medicine is chronically under-stocked.
- `net_stock_flow`: stock received minus what was sold — whether the pharmacy is building up or drawing down buffer.

In [4]:
df["weeks_of_stock"] = df["current_stock"] / (df["sales_last_4_week_average"] + 1)
df["stockout_flag"] = (df["current_stock"] < df["previous_week_sales"]).astype(int)
df["net_stock_flow"] = df["stock_received"] - df["previous_week_sales"]

df[["current_stock", "sales_last_4_week_average", "weeks_of_stock", "stockout_flag", "net_stock_flow"]].head()

,current_stock,sales_last_4_week_average,weeks_of_stock,stockout_flag,net_stock_flow
0,1137,355.00,3.193820,0,26
1,12,355.00,0.033708,1,58
2,26,323.00,0.080247,1,96
3,641,298.00,2.143813,0,83
4,407,287.75,1.409524,0,170


## 4. Momentum / Trend Features

The dataset already provides `previous_week_sales`, `sales_last_2_week`, and `sales_last_4_week_average` — three time horizons of recent demand. The gaps *between* these horizons are informative in their own right: is this week's demand accelerating, decelerating, or flat relative to the recent trend?

In [5]:
df["sales_trend_short"] = df["previous_week_sales"] - df["sales_last_2_week"]
df["sales_trend_long"] = df["previous_week_sales"] - df["sales_last_4_week_average"]
df["sales_acceleration"] = df["sales_last_2_week"] - df["sales_last_4_week_average"]

df[["previous_week_sales", "sales_last_2_week", "sales_last_4_week_average",
    "sales_trend_short", "sales_trend_long", "sales_acceleration"]].head()

,previous_week_sales,sales_last_2_week,sales_last_4_week_average,sales_trend_short,sales_trend_long,sales_acceleration
0,355,355,355.00,0,0.00,0.00
1,355,291,355.00,64,0.00,-64.00
2,291,355,323.00,-64,-32.00,32.00
3,248,291,298.00,-43,-50.00,-7.00
4,257,248,287.75,9,-30.75,-39.75


## 5. Disease Burden

A combined illness signal across the three tracked outbreaks. Useful on its own, and lets the model weigh "total disease pressure this week" against `flu_cases`, `dengue_cases`, `covid_cases` individually (which it can still use to learn category-specific effects, e.g. flu cases driving Fever/Pain Relief demand more than Cardiac demand).

In [6]:
df["total_disease_burden"] = df["flu_cases"] + df["dengue_cases"] + df["covid_cases"]

df[["flu_cases", "dengue_cases", "covid_cases", "total_disease_burden"]].head()

,flu_cases,dengue_cases,covid_cases,total_disease_burden
0,376,19,42,437
1,432,23,27,482
2,362,12,11,385
3,302,14,61,377
4,321,20,104,445


## 6. Autoregressive Demand Lag Features

Demand is highly autocorrelated week to week, so the *actual realized demand from prior weeks* is one of the strongest predictors available. These are built by shifting `next_week_demand` **forward within each medicine group** — so `demand_lag_1` for a given row is the actual demand that materialized one period earlier. Because `.shift()` only looks backward, this uses no information from the row's own target or the future, so there's no leakage.

The first 1–4 weeks of each medicine's history don't have prior actuals yet, so those gaps are filled with `sales_last_4_week_average` (a reasonable stand-in for "no history yet") rather than dropped, to avoid losing early-history rows.

In [7]:
df["demand_lag_1"] = df.groupby("medicine_id")["next_week_demand"].shift(1)
df["demand_lag_2"] = df.groupby("medicine_id")["next_week_demand"].shift(2)
df["demand_rolling_std_4"] = (
    df.groupby("medicine_id")["next_week_demand"]
    .shift(1)
    .rolling(4)
    .std()
    .reset_index(level=0, drop=True)
)

for col in ["demand_lag_1", "demand_lag_2"]:
    df[col] = df[col].fillna(df["sales_last_4_week_average"])
df["demand_rolling_std_4"] = df["demand_rolling_std_4"].fillna(0)

df[["medicine_id", "week_number", "demand_lag_1", "demand_lag_2", "demand_rolling_std_4"]].head(8)

,medicine_id,week_number,demand_lag_1,demand_lag_2,demand_rolling_std_4
0,MED001,1,355.0,355.0,0.000000
1,MED001,2,311.0,355.0,0.000000
2,MED001,3,311.0,311.0,0.000000
3,MED001,4,251.0,311.0,0.000000
4,MED001,5,266.0,251.0,30.923292
5,MED001,6,258.0,266.0,27.037012
6,MED001,7,278.0,258.0,11.586630
7,MED001,8,401.0,278.0,67.336840


## 7. Validation

Confirm no missing values were introduced and check how the new features correlate with `next_week_demand`.

In [8]:
print("Shape:", df.shape)
null_counts = df.isnull().sum()
print("\nColumns with missing values:", null_counts[null_counts > 0].to_dict())

new_cols = [c for c in df.columns if c not in original_cols]
print(f"\n{len(new_cols)} new engineered features:")
print(new_cols)

Shape: (23400, 42)

Columns with missing values: {}

25 new engineered features:
['week_sin', 'week_cos', 'medicine_importance_score', 'cat_Antibiotic', 'cat_Cardiac', 'cat_Diabetes', 'cat_Emergency', 'cat_Fever/Pain Relief', 'cat_Gastrointestinal', 'cat_Respiratory', 'cat_Vitamin/Supplement', 'season_Monsoon', 'season_Normal', 'season_Summer', 'season_Winter', 'weeks_of_stock', 'stockout_flag', 'net_stock_flow', 'sales_trend_short', 'sales_trend_long', 'sales_acceleration', 'total_disease_burden', 'demand_lag_1', 'demand_lag_2', 'demand_rolling_std_4']


In [9]:
numeric_new = [c for c in new_cols if df[c].dtype != "bool" and df[c].nunique() > 2]
df[numeric_new + ["next_week_demand"]].corr()["next_week_demand"].drop("next_week_demand").sort_values(ascending=False)

demand_lag_1                 0.927816
demand_lag_2                 0.924889
demand_rolling_std_4         0.648468
net_stock_flow               0.577277
medicine_importance_score    0.208859
sales_trend_long             0.086022
total_disease_burden         0.082111
sales_trend_short            0.057031
weeks_of_stock               0.047657
week_cos                     0.014857
sales_acceleration           0.006332
week_sin                    -0.004341
Name: next_week_demand, dtype: float64

## 8. Save Engineered Dataset

The original columns are preserved alongside the new features, so this file is a strict superset of the raw dataset and ready to feed into the XGBoost modeling notebook. `medicine_id` and `medicine_name` are kept for traceability but should be excluded (or label-encoded) as direct model inputs at training time — `medicine_importance_score`, `cat_*`, and the engineered features carry the meaningful signal instead.

In [10]:
df.to_csv("../Dataset/smart_pharmacy_dataset_engineered.csv", index=False)
print("Saved:", df.shape)

Saved: (23400, 42)
